# Archive 앞부분 조사·선택 추출 — 위에서 아래로 1회

**소스 ZIP 업로드 → 로컬 검사 → JOB 한 곳 설정 → 네트워크 없는 preflight → 명시적 1회 실행 → 결과 검토 → ZIP 보관** 순서입니다. 별도 plan/선택 JSON 업로드나 아래쪽 새 job 셀은 없습니다. 사용자용 사본은 JOB만 미리 채워 제공할 수 있습니다.

기본 `scan`은 실제 Drive에서 byte0부터 제한된 압축 구간을 읽고 해제하여 regular member의 순서·이름·크기를 기록합니다. **Payload 저장0이지만 네트워크 비용은 발생합니다.** Gzip 임의 위치 점프·이어받기·전체 목록/색인 생성이 아닙니다. `includes`는 저장/선택 필터이며 압축 위치를 바꾸지 않습니다. 같은8MiB를 반복해도 다음8MiB로 전진하지 않습니다.

본문8MiB·60초·logical disk16MiB·총 요청128, Range256KiB와 read1/해제64KiB를 유지합니다. Source 전후 검사를 통과하지 못하면 목록도 untrusted입니다. 모든 결과는 train_ready=false입니다. [준비 계약](../docs/ARCHIVE_PREPARATION.md), [실행 안내](../docs/COLAB_SETUP.md)를 따릅니다.


In [ ]:
import sys, os, io, stat, json, tempfile, zipfile, shutil
from pathlib import Path, PurePosixPath
from google.colab import files

print({"python":sys.version.split()[0],"free_disk_bytes":shutil.disk_usage(".").free})
uploaded = files.upload()  # 여기에는 코드가 들어 있는 source ZIP 한 개만 올립니다.
if len(uploaded)!=1:
    raise ValueError("코드 source ZIP 한 개만 선택하세요.")
source_name, source_bytes = next(iter(uploaded.items()))
if not source_name.lower().endswith(".zip") or len(source_bytes)>64*1024**2:
    raise ValueError("64MiB 이하의 코드 source ZIP이 필요합니다.")
excluded={".git",".venv","data","outputs","checkpoints",".aws",".codex",".agents","__pycache__"}
with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    members=archive.infolist()
    if len(members)>5000 or sum(m.file_size for m in members)>128*1024**2:
        raise ValueError("코드 ZIP의 항목 수/해제 크기 상한을 초과했습니다.")
    code_entries=[m for m in members if m.filename.endswith("rewind_archive/__init__.py")]
    if len(code_entries)!=1:
        inner=[PurePosixPath(m.filename).name for m in members if m.filename.lower().endswith(".zip")][:6]
        if inner:
            raise ValueError("배포 묶음 ZIP입니다. 로컬에서 내부 코드 ZIP "+repr(inner)+"만 꺼내 다시 올리세요. 중첩 자동 해제는 하지 않습니다.")
        raise ValueError("rewind_archive/__init__.py를 포함한 코드 source ZIP이 필요합니다. Notebook/결과 ZIP과 구분하세요.")
    seen=set()
    for m in members:
        p=PurePosixPath(m.filename)
        if p.is_absolute() or ".." in p.parts or "\\" in m.filename or any(":" in x for x in p.parts) or any(ord(c)<32 for c in m.filename) or stat.S_ISLNK(m.external_attr>>16):
            raise ValueError("안전하지 않은 코드 ZIP 경로입니다.")
        key=p.as_posix().casefold().rstrip("/")
        if key in seen:raise ValueError("중복 코드 ZIP 경로입니다.")
        seen.add(key)
    extract_root=Path(tempfile.mkdtemp(prefix="rewind_source_",dir="/content"))
    for m in members:
        parts=PurePosixPath(m.filename).parts
        if not any(p in excluded or p==".env" or p.startswith(".env.") for p in parts):
            archive.extract(m,extract_root)
roots=[p.parent.parent for p in extract_root.rglob("rewind_archive/__init__.py")]
if len(roots)!=1:raise ValueError("단일 프로젝트 source ZIP이 필요합니다.")
PROJECT_ROOT=roots[0]
if not (PROJECT_ROOT/"rewind_archive/workflow.py").is_file():
    raise ValueError("이 notebook과 맞는 최신 source ZIP이 필요합니다(workflow.py 누락).")
if any(name=="rewind_archive" or name.startswith("rewind_archive.") for name in sys.modules):
    raise RuntimeError("이전 모듈이 이미 import되어 있습니다. 기존 결과를 먼저 보관하고 런타임을 재시작한 뒤 새 source ZIP으로 시작하세요.")
os.chdir(PROJECT_ROOT)
sys.path.insert(0,str(PROJECT_ROOT))
from rewind_archive.workflow import NOTEBOOK_API_VERSION
if NOTEBOOK_API_VERSION!=2:raise RuntimeError("Notebook/source API 버전이 다릅니다.")
stamp=PROJECT_ROOT/"SOURCE_REVISION.txt"
print({"source_root":str(PROJECT_ROOT),"source_revision":stamp.read_text().strip() if stamp.exists() else "unversioned; code hashes recorded","notebook_api":NOTEBOOK_API_VERSION})
del uploaded, source_bytes


## 로컬 사전 검사

합성 archive·모의 Drive·localhost 검사입니다. 실제 Google 인증·원본 요청은 하지 않습니다. 실패하면 실제 실행으로 진행하지 않습니다. 이 단계는 모델 학습/GPU 시험과 다릅니다.


In [ ]:
import subprocess
RUN_LOCAL_CHECKS = True
if RUN_LOCAL_CHECKS:
    for pattern in ("test_archive.py","test_drive_metadata.py","test_archive_planning.py","test_archive_scan.py","test_archive_workflow.py"):
        subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-p",pattern,"-v"],cwd=PROJECT_ROOT,check=True)


## 1. 사용자 설정 — 아래 JOB만 편집

첫 실제 작업은 `mode="scan"`, `purpose="prefix_inventory"`, `includes=[]`입니다. 확인한 RGB000의 selection을 채우거나 이미 채운 사용자용 notebook을 사용합니다. 별도 JSON 업로드는 필요 없습니다. Scan은 최대128개 regular header를 기록하고 payload는 저장하지 않습니다.

`extract`의 `discovery`는 limits.max_files를 직접2이하로 설정하고 glob을 명시합니다. `targeted`는 실제 관측 경로를 입력해도 동일 byte0에서 시작합니다. `review`/`review`는 previous_result의 기존 런타임 결과만 검사하며 네트워크를 사용하지 않습니다. 저장한 결과 ZIP을 자동 업로드/풀지는 않습니다.

같은 job_id는 실패해도 다시 실행할 수 없습니다. 같은 설정 fingerprint를 다시 읽으려면 **이 JOB에서 새 job_id와 변경한 reason**을 명시해야 합니다. 설정 변경 후에는 아래 preflight부터 순서대로 실행합니다. 커널 밖 과거 실행 기록까지 자동 인식하지 못하므로 이전 ZIP과 기록도 확인하세요. ID/인증정보를 저장소에 커밋하지 않습니다.


In [ ]:
from dataclasses import asdict
from rewind_archive.contracts import Limits
from rewind_archive.drive_metadata import MetadataLimits

JOB = {
    "job_id":"prefix_scan_001",
    "reason":"archive 앞부분 regular member 순서 조사",
    "mode":"scan",
    "purpose":"prefix_inventory",
    "selection":{
        "schema_version":"drive_part_selection_v1",
        "archive_id":"", "dataset_version":"", "kind":"rgb", "total_parts":None,
        "parts":[]  # [{"index":0,"name":"USER_ARCHIVE.tar.gz000","file_id":"USER_FILE_ID"}]
    },
    "includes":[],
    "limits":asdict(Limits()),
    "metadata_limits":asdict(MetadataLimits()),
    "previous_result":""  # mode/purpose=review일 때 기존 런타임 결과 폴더
}


## 2. 최종 preflight — 원본 요청·인증 없음

Mode/purpose·selection/archive·필터·전송/시간/disk·Range/IO와 동일 작업 경고를 확인합니다. 표시한 설정을 아래 실행이 그대로 사용합니다. 이 셀 뒤에서 JOB를 바꾸면 실행을 거부합니다. Size/version/modifiedTime은 아직 조회하지 않았으며 실제 metadata 성공 후 media 직전에 별도로 표시합니다.

이력이 없는 것이 과거 실행이 없었다는 증거는 아닙니다. 현재 커널 기록과 현재 소스 outputs의 job 기록만 검사합니다. 같은 설정/선택을 읽은 이력이 표시되면 원인·목록 조사 목적·예산을 다시 검토합니다.


In [ ]:
from rewind_archive.workflow import make_preflight, load_history
_ARCHIVE_JOB_HISTORY = globals().get("_ARCHIVE_JOB_HISTORY",{})
_ARCHIVE_JOB_HISTORY.update(load_history(PROJECT_ROOT/"outputs"))
PREFLIGHT = make_preflight(JOB,_ARCHIVE_JOB_HISTORY)
print(json.dumps(PREFLIGHT["summary"],ensure_ascii=False,indent=2))
print({"config_sha256":PREFLIGHT["config_sha256"],"effective_sha256":PREFLIGHT["effective_sha256"],"network_requests":0})


## 3. 명시적 1회 실행 — 기본 비활성

위 요약을 검토한 뒤 **EXECUTE_ONCE=True**로 이 셀을 한 번 실행합니다. Preflight 설정이 현재 JOB와 다르면 인증 전 거부합니다. 시작한 job은 먼저 기록한 뒤 인증→metadata→동일8MiB bounded scan/extract 순서로 진행합니다. 실패해도 자동 재시도하지 않습니다.

Metadata128KiB/30초는 준비8MiB/60초와 별도이며 전후 metadata 비용은 준비 예산에 포함됩니다. 최초 인증 비용·전체 wire/OS/RSS는 측정 밖입니다. Scan도 앞 payload를 읽고 해제하는 비용을 냅니다. Name/glob은 seek가 아니며 새 job은 byte0 재읽기입니다. 실행 뒤에는 아래 결과 보관으로 진행합니다.


In [ ]:
EXECUTE_ONCE = False
from rewind_archive.workflow import run_workflow_job
if JOB["mode"]=="review" or EXECUTE_ONCE:
    current=make_preflight(JOB,_ARCHIVE_JOB_HISTORY)
    if current["config_sha256"]!=PREFLIGHT["config_sha256"]:
        raise RuntimeError("JOB 설정이 바뀌었습니다. Preflight부터 다시 실행하세요.")
if JOB["mode"]=="review":
    # Review 설정도 preflight와 동일해야 한다. 원본/인증 호출은 없다.
    RESULT_ROOT=Path(JOB["previous_result"])
elif EXECUTE_ONCE:
    print("실행 직전 effective 설정:",json.dumps(PREFLIGHT["summary"],ensure_ascii=False,indent=2))
    def create_drive_session():
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import AuthorizedSession, Request
        from requests.adapters import HTTPAdapter
        import urllib3
        if not callable(getattr(urllib3.response.HTTPResponse,"read1",None)):
            raise RuntimeError("urllib3 raw.read1 API가 필요합니다.")
        auth.authenticate_user()
        credentials,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
        credentials.refresh(Request())
        session=AuthorizedSession(credentials,max_refresh_attempts=0)
        session.trust_env=False
        session.mount("https://",HTTPAdapter(max_retries=0))
        return session
    RESULT_ROOT, JOB_RESULT = run_workflow_job(JOB,PREFLIGHT,_ARCHIVE_JOB_HISTORY,PROJECT_ROOT/"outputs",create_drive_session)
    print({"result_root":str(RESULT_ROOT),"status":JOB_RESULT["status"],"stop_reason":JOB_RESULT.get("stop_reason"),"train_ready":False})
else:
    print("실제 인증·metadata·media를 실행하지 않았습니다.")


## 4. 결과 검토 — 원본 요청 없음

`scan`의 저장 파일0개는 의도한 동작입니다. 관측목록의 header/regular ordinal은 tar 안에서 도달한 순서이며 frame 순서가 아닙니다. `payload_complete`는 payload+padding을 소비했다는 뜻이고 `stored`와 다릅니다. Name/size는 header 선언이며 PNG/JSON 내용 검증이 아닙니다. 불완전 payload는 별도로 표시합니다.

`bounded_observation_only`도 전체 목록/seek offset 증명이 아닙니다. Source/unsafe/parser 실패 시 `untrusted_failed_source_or_parser`이므로 목록에서 후속 파일 확보를 확정하지 않습니다. Partial·상한·CRC·목록 cap을 함께 확인합니다. 실패/0sample/목록 조사 후에는 ZIP을 보관하고 검토를 위해 중단합니다. 다음 archive·반대쪽 추출을 자동 시작하지 않습니다.


In [ ]:
from rewind_archive.workflow import review_run
if globals().get("RESULT_ROOT") is not None:
    REVIEW = review_run(RESULT_ROOT)
    # 긴 metadata 응답별 audit는 ZIP 보고서에 유지하고 화면에는 합계만 표시한다.
    if "extraction" in REVIEW:
        REVIEW["extraction"].get("stats",{}).pop("metadata_response_audit",None)
    print(json.dumps(REVIEW,ensure_ascii=False,indent=2))
else:
    print("새 결과가 없습니다. 기존 결과를 보려면 단일 JOB의 mode/purpose=review와 previous_result를 설정하세요.")


## 5. 결과 ZIP 저장·다운로드

현재 작업의 job/metadata/pilot/관측목록·완전 파일만 작은 ZIP으로 보관합니다. 원본이나 소스 전체를 넣지 않습니다. 최대17MiB 결과와 비슷한 별도 임시 ZIP 공간을 사용합니다. 기존 ZIP은 덮어쓰거나 자동 추가하지 않으며 다운로드만 다시 실행할 수 있습니다. 개인 file ID/sample이 포함될 수 있으므로 개인 보관합니다. **런타임을 종료하기 전에 ZIP을 내려받으세요.**


In [ ]:
DOWNLOAD_RESULT_ZIP = False
from rewind_archive.workflow import save_result_zip
if globals().get("RESULT_ROOT") is not None:
    RESULT_ZIP_PATH = save_result_zip(RESULT_ROOT)
    print("result ZIP:",RESULT_ZIP_PATH)
    if DOWNLOAD_RESULT_ZIP:
        files.download(str(RESULT_ZIP_PATH))
else:
    print("저장할 결과가 없습니다.")
